# DashGen AI — Dashboard Generation Pipeline

This notebook demonstrates how DashGen AI converts a natural language description into a structured dashboard specification, and how to render it with Python plotting libraries.

In [ ]:
# !pip install pandas numpy matplotlib seaborn langchain langchain-huggingface

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch

plt.style.use('dark_background')
print("Libraries loaded ✓")

## 1. The Dashboard Specification Schema

DashGen AI returns a structured JSON spec that drives the entire rendering pipeline.

In [ ]:
# Example spec returned by the AI
SAMPLE_SPEC = {
    "title": "Sales Performance Dashboard",
    "subtitle": "Q3 2025 | All Regions",
    "theme": "#3b82f6",
    "kpis": [
        {"label": "Total Revenue",  "value": "$8.4M",  "change": "+14.2%", "trend": "up"},
        {"label": "Orders",         "value": "24,810", "change": "+9.1%",  "trend": "up"},
        {"label": "Avg Order Value","value": "$338",   "change": "+4.7%",  "trend": "up"},
        {"label": "Conv. Rate",     "value": "3.82%",  "change": "-0.12%", "trend": "down"},
    ],
    "charts": [
        {"type": "bar",      "title": "Revenue by Region",
         "labels": ["North","South","East","West","Central"],
         "datasets": [{"label": "Revenue ($K)", "data": [2100,1850,2420,1380,650]}]},
        {"type": "line",     "title": "Monthly Revenue Trend",
         "labels": ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep"],
         "datasets": [
             {"label": "2025", "data": [780,820,910,950,1020,1080,1150,1180,510]},
             {"label": "2024", "data": [680,710,790,820,870,940,980,1020,450]},
         ]},
        {"type": "pie",      "title": "Revenue by Category",
         "labels": ["Category A","Category B","Category C","Category D"],
         "datasets": [{"label": "Revenue", "data": [38,27,21,14]}]},
    ],
    "table": {
        "title": "Top Products",
        "headers": ["Product","Revenue","Growth","Margin"],
        "rows": [
            ["Alpha","$1.82M","+18.4%","42%"],
            ["Beta", "$1.44M","+12.1%","38%"],
            ["Gamma","$1.21M","+8.7%", "45%"],
        ]
    }
}
print("Schema loaded ✓")
print(f"KPIs: {len(SAMPLE_SPEC['kpis'])}")
print(f"Charts: {len(SAMPLE_SPEC['charts'])}")

## 2. Python Dashboard Renderer

In [ ]:
def render_dashboard(spec):
    theme = spec.get("theme", "#3b82f6")
    charts = spec.get("charts", [])
    kpis   = spec.get("kpis", [])
    nc     = len(charts)

    fig = plt.figure(figsize=(16, 10), facecolor='#0f1420')
    fig.suptitle(spec.get("title","Dashboard"), fontsize=18, fontweight='bold',
                 color='white', y=0.98)

    # KPI row
    for i, kpi in enumerate(kpis[:4]):
        ax = fig.add_axes([0.02 + i*0.245, 0.82, 0.22, 0.12])
        ax.set_facecolor('#1e2a42')
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values(): spine.set_color('#374151')
        color = '#34d399' if kpi['trend'] == 'up' else '#f87171'
        arrow = '▲' if kpi['trend'] == 'up' else '▼'
        ax.text(0.5, 0.72, kpi['value'], transform=ax.transAxes, ha='center',
                fontsize=20, fontweight='bold', color=theme)
        ax.text(0.5, 0.42, kpi['label'], transform=ax.transAxes, ha='center',
                fontsize=10, color='#94a3b8')
        ax.text(0.5, 0.15, f"{arrow} {kpi['change']}", transform=ax.transAxes,
                ha='center', fontsize=11, color=color)

    COLORS = [theme, '#10b981', '#f59e0b', '#ef4444', '#8b5cf6']

    for i, ch in enumerate(charts[:3]):
        col  = i % 2
        row  = i // 2
        ax = fig.add_axes([0.02 + col*0.49, 0.45 - row*0.38, 0.46, 0.33])
        ax.set_facecolor('#161d2e')
        for spine in ax.spines.values(): spine.set_color('#374151')
        ax.tick_params(colors='#94a3b8', labelsize=9)
        ax.set_title(ch['title'], color='#94a3b8', fontsize=11, pad=8)

        labels = ch.get('labels', [])
        for di, ds in enumerate(ch.get('datasets', [])):
            c = COLORS[di % len(COLORS)]
            if ch['type'] == 'bar':
                x = np.arange(len(labels))
                w = 0.8 / len(ch['datasets'])
                ax.bar(x + di*w, ds['data'], w*0.9, color=c+'cc', label=ds.get('label',''))
                ax.set_xticks(x + 0.4); ax.set_xticklabels(labels, rotation=15, ha='right')
            elif ch['type'] == 'line':
                ax.plot(labels, ds['data'], color=c, linewidth=2.5, marker='o',
                        markersize=4, label=ds.get('label',''))
                ax.fill_between(range(len(labels)), ds['data'], alpha=0.1, color=c)
            elif ch['type'] in ('pie','doughnut'):
                wedge_c = COLORS[:len(labels)]
                ax.pie(ds['data'], labels=labels, colors=wedge_c, autopct='%1.0f%%',
                       pctdistance=0.8, textprops={'color':'#94a3b8','fontsize':9})
                if ch['type'] == 'doughnut':
                    centre_circle = plt.Circle((0,0), 0.5, fc='#161d2e')
                    ax.add_artist(centre_circle)
        ax.grid(axis='y', color='#1e2a42', alpha=0.5) if ch['type'] != 'pie' else None
        if len(ch.get('datasets',[])) > 1:
            ax.legend(fontsize=9, framealpha=0, labelcolor='#94a3b8')

    plt.savefig('../diagrams/dashgen_sample.png', dpi=110, bbox_inches='tight',
                facecolor='#0f1420')
    plt.show()
    print("Dashboard rendered and saved to diagrams/dashgen_sample.png")

render_dashboard(SAMPLE_SPEC)

## 3. LLM Spec Generation (live)

In production, the spec is generated by the LLM. This block shows how the prompt is constructed and the response parsed.

In [ ]:
# This requires: pip install langchain langchain-huggingface
# and export HF_API_TOKEN=hf_xxx

prompt_template = """You are DashGen AI. Generate a dashboard spec as JSON.

User request: "{description}"

Return ONLY valid JSON:
{{
  "title": "...",
  "subtitle": "...",
  "theme": "{theme}",
  "kpis": [...],
  "filters": [...],
  "charts": [...],
  "table": {{...}}
}}"""

# Live generation example (uncomment to run):
"""
import os
from langchain_huggingface import HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    huggingfacehub_api_token=os.environ["HF_API_TOKEN"],
    max_new_tokens=1500,
    temperature=0.15,
)

description = "Healthcare operations dashboard for a hospital"
prompt = prompt_template.format(description=description, theme="#059669")
raw = llm.invoke(prompt)

import json, re
clean = re.sub(r"```(?:json)?\n?", "", raw).replace("```","").strip()
spec  = json.loads(clean[clean.find("{"):clean.rfind("}")+1])
render_dashboard(spec)
"""
print("Uncomment block above to generate a live dashboard with HuggingFace LLM.")
print("The DashGen frontend HTML app works without this step using the Claude API.")

## Summary

| Component | Implementation |
|---|---|
| NL → JSON | LLM (HuggingFace Mistral / Claude) |
| Spec schema | Pydantic-validated JSON |
| Rendering (browser) | Chart.js + vanilla JS |
| Rendering (Python) | Matplotlib + GridSpec |
| CSV support | PapaParse (browser) / pandas (Python) |
| API | FastAPI (dashgen.py, port 8004) |